# 📊 Data Quality Observability & Intelligent Remediation
**Autonomous AI-Agent Platform for Enterprise Data Health**

▶️ Run the cell below — the app launches automatically. No manual steps required.

---
### 🔑 Optional: Enable Email Alerts
Add credentials via **Colab Secrets** (🔑 icon in the left sidebar) before running:

| Secret Name | Description |
|---|---|
| `SENDGRID_API_KEY` | Your SendGrid API key |
| `DQ_ALERT_RECIPIENTS` | Comma-separated recipient emails |
| `NGROK_AUTH_TOKEN` | *(Optional)* ngrok token for a stable tunnel |

The platform runs fully without any of these — email and AI features are skipped gracefully.

In [ ]:
# @title ▶️ Launch Platform (Run Me)
# ═══════════════════════════════════════════════════════════════════════════════
# Data Quality Observability & Intelligent Remediation — Colab Launcher
# Zero-touch: clones repo → generates datasets → installs deps → launches app
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess, threading, time, textwrap, sys

# ── Step 1: Clone / update repository ────────────────────────────────────────
REPO_URL    = "https://github.com/Teja-Jan/Data-Quality-Observability-Intelligent-Remediation.git"
REPO_BRANCH = "Data-Quality-Observability-and-Intelligent-Remediation"
REPO_DIR    = "Data-Quality-Observability-Intelligent-Remediation"

if not os.path.exists(REPO_DIR):
    print("📥 Cloning repository...")
    result = subprocess.run(
        ["git", "clone", "-b", REPO_BRANCH, "--depth", "1", REPO_URL],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("❌ Clone failed:", result.stderr)
        raise RuntimeError("Repository clone failed")
    print("✅ Repository cloned.")
else:
    print("✅ Repository already present — pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                   capture_output=True)

os.chdir(REPO_DIR)
print(f"📂 Working directory: {os.getcwd()}")

# ── Step 2: Install Python dependencies ──────────────────────────────────────
print("\n📦 Installing dependencies (this takes ~60 s first time)...")
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    capture_output=True, text=True
)
if r.returncode != 0:
    print("⚠️ Some packages may have failed:", r.stderr[-500:])
else:
    print("✅ Dependencies installed.")

# Install pyngrok for tunnelling
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok>=6.0"],
               capture_output=True)

# ── Step 3: Generate demo datasets (data/ is gitignored — must create here) ──
DATA_RAW = os.path.join(os.getcwd(), "data", "raw")
DOMAINS   = ["healthcare", "finance", "insurance", "supply_chain", "automotive"]
needs_gen = any(
    not os.path.exists(os.path.join(DATA_RAW, f"{d}_dataset.csv"))
    for d in DOMAINS
)

if needs_gen:
    print("\n📊 Generating synthetic demo datasets (one-time, ~30 s)...")
    gen_result = subprocess.run(
        [sys.executable, "src/data_generation/generate_datasets.py"],
        capture_output=True, text=True
    )
    if gen_result.returncode == 0:
        print("✅ All 5 domain datasets generated.")
    else:
        print("⚠️ Dataset generation warning:", gen_result.stderr[-300:])
        # Fallback: generate minimal datasets inline so app always has data
        print("⚙️  Creating minimal fallback datasets...")
        import pandas as pd, numpy as np, random
        os.makedirs(DATA_RAW, exist_ok=True)
        for d in DOMAINS:
            p = os.path.join(DATA_RAW, f"{d}_dataset.csv")
            if not os.path.exists(p):
                n = 500
                df = pd.DataFrame({
                    "id":           [f"{d.upper()[:3]}-{i:05d}" for i in range(n)],
                    "name":         [f"Record {i}" for i in range(n)],
                    "value":        np.random.uniform(100, 10000, n).round(2),
                    "status":       random.choices(["Active","Inactive","Pending"], k=n),
                    "date":         pd.date_range("2022-01-01", periods=n, freq="D").astype(str),
                    "last_updated": pd.date_range("2023-01-01", periods=n, freq="D").astype(str),
                    "email":        [f"user{i}@example.com" if i % 10 != 0 else None for i in range(n)],
                })
                df.to_csv(p, index=False)
        print("✅ Fallback datasets created.")
else:
    print("\n✅ Demo datasets already exist — skipping generation.")

# ── Step 4: Write .env (Colab Secrets → env vars → .env file) ────────────────
print("\n🔑 Loading credentials from Colab Secrets (optional)...")
SG_KEY, SG_RCPT, NGROK_TOKEN = "", "", ""
try:
    from google.colab import userdata
    SG_KEY     = userdata.get("SENDGRID_API_KEY")    or ""
    SG_RCPT    = userdata.get("DQ_ALERT_RECIPIENTS") or ""
    NGROK_TOKEN = userdata.get("NGROK_AUTH_TOKEN")   or ""
    if SG_KEY:
        print("  ✅ SendGrid key found.")
    else:
        print("  ℹ️  No SENDGRID_API_KEY — email alerts disabled.")
except Exception:
    print("  ℹ️  Colab Secrets not accessible — credentials skipped.")

EMAIL_PROVIDER = "sendgrid" if SG_KEY else "smtp"

env_text = textwrap.dedent(f"""
    EMAIL_PROVIDER={EMAIL_PROVIDER}
    SENDGRID_API_KEY={SG_KEY}
    DQ_ALERT_RECIPIENTS={SG_RCPT}
    APP_NAME=Data Quality Observability & Intelligent Remediation
    APP_ENV=colab
    LOG_LEVEL=INFO
    DB_PATH=src/db/dq_metadata.db
    AUTO_FIX_MIN_RESOLUTIONS=3
    AUTO_FIX_CONFIDENCE_THRESHOLD=0.95
    DEFAULT_CONN_TYPE=
""").strip()

with open(".env", "w") as f:
    f.write(env_text)
print("✅ .env written.")

# ── Step 5: Ollama in background (non-blocking — app launches while it installs)
def _setup_ollama():
    try:
        os.system("curl -fsSL https://ollama.com/install.sh | sh >/dev/null 2>&1")
        os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
        os.system("ollama serve >/dev/null 2>&1 &")
        time.sleep(6)
        os.system("ollama pull llama3 >/dev/null 2>&1")
    except Exception:
        pass  # AI chat falls back to rule-based mode

threading.Thread(target=_setup_ollama, daemon=True).start()
print("\n🤖 Ollama installing in background — AI features activate when ready.")

# ── Step 6: Launch Streamlit ──────────────────────────────────────────────────
PORT = 8501
print("\n🚀 Starting Streamlit...")
proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "src/app.py",
        "--server.port", str(PORT),
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.fileWatcherType", "none",
        "--browser.gatherUsageStats", "false",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Wait up to 45 s for Streamlit to be ready
print("⏳ Waiting for Streamlit", end="", flush=True)
import urllib.request
ready = False
for _ in range(45):
    time.sleep(1)
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/_stcore/health", timeout=1)
        ready = True
        break
    except Exception:
        print(".", end="", flush=True)
print(" Ready!" if ready else " (timeout — continuing anyway)")

# ── Step 7: Tunnel ─────────────────────────────────────────────────────────────
# Primary: Colab's built-in port output (most reliable, no auth needed)
tunnel_url = None
try:
    from google.colab.output import eval_js
    tunnel_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
    print(f"\n✅ Colab proxy tunnel active.")
except Exception:
    pass

# Fallback: pyngrok
if not tunnel_url:
    try:
        from pyngrok import ngrok
        if NGROK_TOKEN:
            ngrok.set_auth_token(NGROK_TOKEN)
            print("  ✅ ngrok authenticated with token.")
        public = ngrok.connect(PORT, "http")
        tunnel_url = public.public_url
        print(f"✅ ngrok tunnel active.")
    except Exception as e:
        print(f"⚠️  Tunnel error: {e}")
        tunnel_url = f"http://localhost:{PORT}"

# ── Step 8: Display URL ───────────────────────────────────────────────────────
print("\n" + "═" * 68)
print("  ✅  Data Quality Observability & Intelligent Remediation")
print("═" * 68)
print(f"  🌐  App URL:  {tunnel_url}")
print("═" * 68)
print("  ℹ️   Keep this cell running. Interrupt kernel to stop.")
print("═" * 68 + "\n")

# ── Step 9: Stream logs ───────────────────────────────────────────────────────
try:
    for line in proc.stdout:
        stripped = line.strip()
        if stripped and not stripped.startswith("  "):
            print(stripped)
except KeyboardInterrupt:
    print("\n🛑 Shutting down...")
    proc.terminate()
    try:
        from pyngrok import ngrok
        ngrok.kill()
    except Exception:
        pass